In [1]:
from tucker_tensor import TuckerDecomposition, np_sim
from time import time
start = time()
tensor = TuckerDecomposition.load_from_disk(dataset="fineweb_large",
                                            method="sc",
                                            dims=1000,
                                            rank=150,
                                            iterations=1000,
                                            map_location="cpu",
                                            name = "kl"

                              )
print(f"Loaded Tucker decomposition in {time() - start:.4f} seconds.")


2025-12-10 14:31:09.706473: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Loaded Tucker decomposition with the following parameters:
  timestamp: 2025-12-10T12:40:29.760414Z
  dataset: fineweb_dutch
  input_vectors: 10000000
  method: sc
  dimensionality: 1000
  rank: [150, 150, 150]
  max_iterations: 1000
  iterations: 43
  final_error: 5.446615219116211
  final_fitness: 0.009505713963587731
  tolerance: 1e-06
  random_state: 1
  output_tensor: kl_sc_1000d_150r_1000i.pt
  runtime_seconds: 124.49
Loaded Tucker decomposition in 0.4053 seconds.


In [3]:
target_tuple = ("voeren", "generaal", "oorlog")
tensor.get_expected_element(target_tuple, "verb")
print("~" * 20)
tensor.get_expected_element(target_tuple, "subject")
print("~" * 20)
tensor.get_expected_element(target_tuple, "object")

Top 5 most similar verbs to the integrated core tensor:
Subject: voeren, Similarity: 0.7223202586174011, Cosine sim with target verb activations: 1.0
Subject: uitvoeren, Similarity: 0.7219336032867432, Cosine sim with target verb activations: 0.9983816742897034
Subject: verrichten, Similarity: 0.6879570484161377, Cosine sim with target verb activations: 0.9638484120368958
Subject: aanvoeren, Similarity: 0.6412767171859741, Cosine sim with target verb activations: 0.9000706672668457
Subject: financieren, Similarity: 0.34369122982025146, Cosine sim with target verb activations: 0.011398770846426487
~~~~~~~~~~~~~~~~~~~~
Top 5 most similar subjects to the integrated core tensor:
Subject: Rusland, Similarity: 0.5409570336341858, Cosine sim with target subject activations: 0.04053357243537903
Subject: Fransen, Similarity: 0.5329156517982483, Cosine sim with target subject activations: 0.05092141777276993
Subject: Washington, Similarity: 0.5134420990943909, Cosine sim with target subject acti

In [4]:
target_tuple = ("spelen", "kind", "spel")
tensor.get_expected_element(target_tuple, "verb")
print("~" * 20)
tensor.get_expected_element(target_tuple, "subject")
print("~" * 20)
tensor.get_expected_element(target_tuple, "object")

Top 5 most similar verbs to the integrated core tensor:
Subject: vinden, Similarity: 0.631986677646637, Cosine sim with target verb activations: 3.1089586200599983e-12
Subject: uitvoeren, Similarity: 0.4896640479564667, Cosine sim with target verb activations: 4.319885161391257e-12
Subject: voeren, Similarity: 0.4891347289085388, Cosine sim with target verb activations: 2.6596693597502608e-12
Subject: verrichten, Similarity: 0.4667699933052063, Cosine sim with target verb activations: 1.0822966654833177e-11
Subject: aanvoeren, Similarity: 0.4335440397262573, Cosine sim with target verb activations: 2.2560269624660734e-11
~~~~~~~~~~~~~~~~~~~~
Top 5 most similar subjects to the integrated core tensor:
Subject: Verstappen, Similarity: 0.5838361382484436, Cosine sim with target subject activations: 0.00278268288820982
Subject: wereldkampioen, Similarity: 0.5536702871322632, Cosine sim with target subject activations: 0.008252093568444252
Subject: Max, Similarity: 0.5155972242355347, Cosine

In [28]:
import csv
from utils import DATA_DIR
vector_path = DATA_DIR/"vectors/fineweb_dutch_10000000.csv"
def load_og_sentences(vector_path):
    sentences = set()
    with open(vector_path, "r", encoding="utf-8") as f:
        reader = csv.reader(f)
        next(reader)  # skip header
        for row in reader:
            sent_id, vector, sentence = row[0], row[1], row[2]
            # the vectors have form "('verb', 'subject', 'object', 'rest', 'rest')"
            # we interpret the vector as a tuple of (verb, subject, object, *rest)
            vector = eval(vector)
            v, s, o = vector[0], vector[1], vector[2]
            sentences.add((v, s, o))
    return list(sentences)
og_sentences = load_og_sentences(vector_path)
print(f"Loaded {len(og_sentences)} original sentences.")

Loaded 3923812 original sentences.


In [49]:
og_sentences[:5]

[('wrijven', 'Philip', None),
 ('zijn', 'bakkerij', None),
 ('vlieg', 'ze', None),
 ('geven', 'headingsMap', 'overzicht'),
 ('rolt', 'Salon', None)]

In [9]:
from tucker_tensor import TuckerDecomposition
tensor = TuckerDecomposition.load_from_disk(dataset="fineweb_sparse",
                              method="sc",
                              dims=2000,
                              rank=150,
                              iterations=10000,
                              map_location="cpu")

In [39]:
from tqdm import tqdm
import random

def evaluate_sample(tensor, sentences, n_samples=100, seed=42):
    total_score = 0
    random.seed(seed)
    sampled_sentences = random.sample(sentences, n_samples)
    for tup in tqdm(sampled_sentences):
        if not tensor.check_vocab(tup):
            continue
        score = 0
        for i, element in enumerate(tup):
            role = ["verb", "subject", "object"][i]
            r2i = {"verb": "v2i", "subject": "s2i", "object": "o2i"}[role]
            G_excluded = tensor.excluded_role_vector(tup, role=role)
            F = tensor.factors[i].cpu().numpy()    # (N,R)
            similarities = F @ G_excluded / (np.linalg.norm(F, axis=1) * np.linalg.norm(G_excluded))
            # we print the rank of the actual element
            idx = tensor.vocab[r2i][element]
            rank = np.sum(similarities >= similarities[idx])
            score += 1/rank
        total_score += score/len(tup)
    average_score = total_score / n_samples
    print(f"Average expected role vector rank score over {n_samples} samples: {average_score}")
    return average_score


In [36]:
evaluate_sample(tensor, og_sentences, n_samples=10000, seed=42)

Evaluating samples: 100%|██████████| 10000/10000 [00:11<00:00, 842.66it/s]

Average expected role vector rank score over 10000 samples: 0.018945336950203787


0.018945336950203787

In [38]:
evaluate_sample(tensor, og_sentences, n_samples=30000, seed=10)

Evaluating samples: 100%|██████████| 30000/30000 [00:33<00:00, 892.46it/s] 

Average expected role vector rank score over 30000 samples: 0.019391318638944995


0.019391318638944995

In [45]:
methods = ["counting", "sc"]
dims = [1000,1500,2000,2500]
ranks = [100, 150]
iters = 10000
eval_scores = {}
for method in methods:
    for dim in dims:
        for rank in ranks:
            print(f"Evaluating method: {method}, dims: {dim}, rank: {rank}")
            tensor = TuckerDecomposition.load_from_disk(dataset="fineweb_sparse",
                                          method=method,
                                          dims=dim,
                                          rank=rank,
                                          iterations=iters,
                                          map_location="cpu")
            score = evaluate_sample(tensor, og_sentences, n_samples=10000, seed=2)
            eval_scores[(method, dim, rank)] = score

Evaluating method: counting, dims: 1000, rank: 100


100%|██████████| 10000/10000 [00:04<00:00, 2220.55it/s]


Average expected role vector rank score over 10000 samples: 0.008005567249444605
Evaluating method: counting, dims: 1000, rank: 150


100%|██████████| 10000/10000 [00:12<00:00, 770.53it/s]


Average expected role vector rank score over 10000 samples: 0.008135747832486892
Evaluating method: counting, dims: 1500, rank: 100


...

Average expected role vector rank score over 10000 samples: 0.01775513345351772
Evaluating method: sc, dims: 2500, rank: 100


100%|██████████| 10000/10000 [00:06<00:00, 1523.66it/s]


Average expected role vector rank score over 10000 samples: 0.017900272862329602
Evaluating method: sc, dims: 2500, rank: 150


100%|██████████| 10000/10000 [00:20<00:00, 495.51it/s]

Average expected role vector rank score over 10000 samples: 0.016818930386521467


In [46]:
eval_scores

{('counting', 1000, 100): 0.008005567249444605,
 ('counting', 1000, 150): 0.008135747832486892,
 ('counting', 1500, 100): 0.0058814223401221265,
 ('counting', 1500, 150): 0.010463867494784462,
 ('counting', 2000, 100): 0.007303689323505712,
 ('counting', 2000, 150): 0.00738882676741093,
 ('counting', 2500, 100): 0.005794407214817209,
 ('counting', 2500, 150): 0.007813628437792906,
 ('sc', 1000, 100): 0.013187061799140825,
 ('sc', 1000, 150): 0.013142257480186666,
 ('sc', 1500, 100): 0.013505195471523315,
 ('sc', 1500, 150): 0.015625856969649347,
 ('sc', 2000, 100): 0.016916022770005566,
 ('sc', 2000, 150): 0.01775513345351772,
 ('sc', 2500, 100): 0.017900272862329602,
 ('sc', 2500, 150): 0.016818930386521467}

In [47]:
methods = ["counting", "sc", "sii"]
datasets = ["fineweb_sparse", "fineweb_large"]
dims = [1000,1500,2000]
ranks = [100, 150]
eval_scores = {}
for method in methods:
    for dim in dims:
        for rank in ranks:
            for dataset in datasets:
                 print(f"Evaluating method: {method}, dims: {dim}, rank: {rank}, dataset: {dataset}")
                 tensor = TuckerDecomposition.load_from_disk(dataset=dataset,
                                               method=method,
                                               dims=dim,
                                               rank=rank,
                                               iterations=2500 if dataset=="fineweb_large" else 10000,
                                               map_location="cpu")
                 score = evaluate_sample(tensor, og_sentences, n_samples=10000, seed=2)
                 eval_scores[(method, dim, rank, dataset)] = score

Evaluating method: counting, dims: 1000, rank: 100, dataset: fineweb_sparse


100%|██████████| 10000/10000 [00:04<00:00, 2298.54it/s]


Average expected role vector rank score over 10000 samples: 0.008005567249444605
Evaluating method: counting, dims: 1000, rank: 100, dataset: fineweb_large


100%|██████████| 10000/10000 [00:05<00:00, 1939.59it/s]


Average expected role vector rank score over 10000 samples: 0.008365408178050836
Evaluating method: counting, dims: 1000, rank: 150, dataset: fineweb_sparse


...

100%|██████████| 10000/10000 [00:17<00:00, 577.80it/s]


Average expected role vector rank score over 10000 samples: 0.01775513345351772
Evaluating method: sc, dims: 2000, rank: 150, dataset: fineweb_large


100%|██████████| 10000/10000 [00:19<00:00, 525.73it/s]

Average expected role vector rank score over 10000 samples: 0.028377065645491462
Evaluating method: sii, dims: 1000, rank: 100, dataset: fineweb_sparse


FileNotFoundError: Missing decomposition file: /home/local/stefa/data/tensors/fineweb_sparse/decomposition/sii_1000d_100r_10000i.pt

In [48]:
eval_scores

{('counting', 1000, 100, 'fineweb_sparse'): 0.008005567249444605,
 ('counting', 1000, 100, 'fineweb_large'): 0.008365408178050836,
 ('counting', 1000, 150, 'fineweb_sparse'): 0.008135747832486892,
 ('counting', 1000, 150, 'fineweb_large'): 0.006337942242581181,
 ('counting', 1500, 100, 'fineweb_sparse'): 0.0058814223401221265,
 ('counting', 1500, 100, 'fineweb_large'): 0.00928529449753193,
 ('counting', 1500, 150, 'fineweb_sparse'): 0.010463867494784462,
 ('counting', 1500, 150, 'fineweb_large'): 0.008975236604856182,
 ('counting', 2000, 100, 'fineweb_sparse'): 0.007303689323505712,
 ('counting', 2000, 100, 'fineweb_large'): 0.00865091907878553,
 ('counting', 2000, 150, 'fineweb_sparse'): 0.00738882676741093,
 ('counting', 2000, 150, 'fineweb_large'): 0.007141084341113361,
 ('sc', 1000, 100, 'fineweb_sparse'): 0.013187061799140825,
 ('sc', 1000, 100, 'fineweb_large'): 0.01762527951990964,
 ('sc', 1000, 150, 'fineweb_sparse'): 0.013142257480186666,
 ('sc', 1000, 150, 'fineweb_large'): 0

# Development code for excluded role vector and similarity checks

In [37]:
import numpy as np

start = time()
target_tuple = ("voeren", "president", "oorlog")
assert tensor.check_vocab(target_tuple), "Target tuple not in vocabularies"
v, s, o = tensor.fetch_latents(target_tuple)
G_verb = tensor.excluded_role_vector(target_tuple, role="verb")
print("Core tensor after integrating subject and object activations:", G_verb.shape)
cos_sim = np_sim(G_verb, v)
print("Cosine similarity between G_verb and verb latent factor:", cos_sim)
V = tensor.factors[0].cpu().numpy()    # (Nv,R)
similarities = V @ G_verb / (np.linalg.norm(V, axis=1) * np.linalg.norm(G_verb))
# we get the top 5 most similar verbs
k = 5
top_k_indices = np.argsort(similarities)[-k:][::-1]
print(f"Top {k} most similar verbs to the integrated core tensor:")
for idx in top_k_indices:
    verb_str = [k for k, v in tensor.vocab["v2i"].items() if v == idx][0]
    # we also check for the similarity between the verb factor activations for this verb, and those from the target tuple verb
    verb_act = tensor.factors[0][idx, :].cpu().numpy()
    cos_sim = np_sim(verb_act, v)
    print(f"Verb: {verb_str}, Similarity: {similarities[idx]}, Cosine sim with target verb activations: {cos_sim}")
print(f"Total time: {time() - start:.4f} seconds.")

Core tensor after integrating subject and object activations: (100,)
Cosine similarity between G_verb and verb latent factor: 0.9634987
Top 5 most similar verbs to the integrated core tensor:
Verb: voeren, Similarity: 0.9634987115859985, Cosine sim with target verb activations: 1.0
Verb: ondernemen, Similarity: 0.9477770924568176, Cosine sim with target verb activations: 0.9558386206626892
Verb: aangaan, Similarity: 0.7107551693916321, Cosine sim with target verb activations: 0.7167853713035583
Verb: aanvoeren, Similarity: 0.702592670917511, Cosine sim with target verb activations: 0.7055646777153015
Verb: veroordelen, Similarity: 0.7021585702896118, Cosine sim with target verb activations: 0.6405983567237854
Total time: 0.0029 seconds.


In [38]:
# we assert all are in vocab
from tucker_tensor import _to_np
for word, key in zip(target_tuple, ["v2i", "s2i", "o2i"]):
    assert word in tensor.vocab[key], f"Word '{word}' not in vocab '{key}'"
tensor.check_vocab(target_tuple)
ids = [tensor.vocab["v2i"][target_tuple[0]],
       tensor.vocab["s2i"][target_tuple[1]],
       tensor.vocab["o2i"][target_tuple[2]]]
print("Indices in vocabularies:", ids)
verb_rep = tensor.factors[0][ids[0], :]
print(f"Latent factor vector for verb '{target_tuple[0]}':", verb_rep.shape)

# we retrieve the activations for subject and object
subj_rep = tensor.factors[1][ids[1], :]
obj_rep = tensor.factors[2][ids[2], :]
print(f"Latent factor vector for subject '{target_tuple[1]}':", subj_rep.shape)
print(f"Latent factor vector for object '{target_tuple[2]}':", obj_rep.shape)

# we multiply only subject and object into the core
G = _to_np(tensor.core)
G_verb = np.einsum('pqr,q,r->p', G, subj_rep.cpu().numpy(), obj_rep.cpu().numpy())
print("Core tensor after integrating subject and object activations:", G_verb.shape)


# we check the similarity between G_verb and the verb latent factor
cos_sim = np.dot(G_verb, verb_rep.cpu().numpy()) / (np.linalg.norm(G_verb) * np.linalg.norm(verb_rep.cpu().numpy()))
print("Cosine similarity between G_verb and verb latent factor:", cos_sim)

V = tensor.factors[0].cpu().numpy()    # (Nv,R)
similarities = V @ G_verb / (np.linalg.norm(V, axis=1) * np.linalg.norm(G_verb))
# we get the top 5 most similar verbs
k = 5
top_k_indices = np.argsort(similarities)[-k:][::-1]
print(f"Top {k} most similar verbs to the integrated core tensor:")
for idx in top_k_indices:
    verb_str = [k for k, v in tensor.vocab["v2i"].items() if v == idx][0]
    # we also check for the similarity between the verb factor activations for this verb, and those from the target tuple verb
    verb_act = tensor.factors[0][idx, :].cpu().numpy()
    cos_sim = np.dot(verb_act, verb_rep.cpu().numpy()) / (np.linalg.norm(verb_act) * np.linalg.norm(verb_rep.cpu().numpy()))
    print(f"Verb: {verb_str}, Similarity: {similarities[idx]}, Cosine sim with target verb activations: {cos_sim}")

Indices in vocabularies: [112, 28, 225]
Latent factor vector for verb 'voeren': torch.Size([100])
Latent factor vector for subject 'president': torch.Size([100])
Latent factor vector for object 'oorlog': torch.Size([100])
Core tensor after integrating subject and object activations: (100,)
Cosine similarity between G_verb and verb latent factor: 0.9634987
Top 5 most similar verbs to the integrated core tensor:
Verb: voeren, Similarity: 0.9634987115859985, Cosine sim with target verb activations: 1.0
Verb: ondernemen, Similarity: 0.9477770924568176, Cosine sim with target verb activations: 0.9558386206626892
Verb: aangaan, Similarity: 0.7107551693916321, Cosine sim with target verb activations: 0.7167853713035583
Verb: aanvoeren, Similarity: 0.702592670917511, Cosine sim with target verb activations: 0.7055646777153015
Verb: veroordelen, Similarity: 0.7021585702896118, Cosine sim with target verb activations: 0.6405983567237854


# we can use this to assess the quality of our tensor
For a held-out set of triples, we can compute the excluded role vector for each role, and check if the correct word is among the top k most similar words to the excluded role vector.

In [ ]:
def el_rank(tucker, target_tuple):
    score = 0
    for i, element in enumerate(target_tuple):
        role = ["verb", "subject", "object"][i]
        r2i = _voc_index(role)
        g_excluded = tucker.excluded_role_vector(target_tuple, role=role)
        factor = tucker.factors[i].cpu().numpy()  # (N,R)
        similarities = factor @ g_excluded / (np.linalg.norm(factor, axis=1) * np.linalg.norm(g_excluded))
        # we print the rank of the actual element
        idx = tucker.vocab[r2i][element]
        rank = np.sum(similarities >= similarities[idx])
        score += 1 / rank
    return score / len(target_tuple)

In [54]:
test_set = [("voeren", "president", "oorlog"),
            ("spelen", "kind", "bal"),
            ("lezen", "student", "boek"),
            ("rijden", "chauffeur", "auto")]
for tup in test_set:
    assert tensor.check_vocab(tup), f"Tuple {tup} not in vocabularies"

score = 0
for tup in test_set:
    score += el_rank(tensor, tup)
print(f"Average EL rank score over test set: {score/len(test_set)}")

Average EL rank score over test set: 0.13966806829625197


In [55]:
methods = ["sc", "counting"]
dims = [1000,1500,2000,2500]
iters = 1000
for method in methods:
    for dim in dims:
        tensor = TuckerDecomposition.load_from_disk(dataset="fineweb_sparse",
                                      method=method,
                                      dims=dim,
                                      rank=100,
                                      iterations=iters,
                                      map_location="cpu")
        score = 0
        for tup in test_set:
            score += tensor.el_rank(tup)
        print(f"Method: {method}, Dims: {dim}, Iters: {iters}, Average EL rank score over test set: {score/len(test_set)}")

Method: sc, Dims: 1000, Iters: 1000, Average EL rank score over test set: 0.08820063094235615
Method: sc, Dims: 1500, Iters: 1000, Average EL rank score over test set: 0.24760692354765493
Method: sc, Dims: 2000, Iters: 1000, Average EL rank score over test set: 0.030531533923291318
Method: sc, Dims: 2500, Iters: 1000, Average EL rank score over test set: 0.019969672143777498
Method: counting, Dims: 1000, Iters: 1000, Average EL rank score over test set: 0.005368167240753407
Method: counting, Dims: 1500, Iters: 1000, Average EL rank score over test set: 0.005379332375561909
Method: counting, Dims: 2000, Iters: 1000, Average EL rank score over test set: 0.004533611392679765
Method: counting, Dims: 2500, Iters: 1000, Average EL rank score over test set: 0.0026345023841195503


In [56]:
iters = 10000
for method in methods:
    for dim in dims:
        tensor = TuckerDecomposition.load_from_disk(dataset="fineweb_sparse",
                                      method=method,
                                      dims=dim,
                                      rank=100,
                                      iterations=iters,
                                      map_location="cpu")
        score = 0
        for tup in test_set:
            score += tensor.el_rank(tup)
        print(f"Method: {method}, Dims: {dim}, Iters: {iters}, Average EL rank score over test set: {score/len(test_set)}")

Method: sc, Dims: 1000, Iters: 10000, Average EL rank score over test set: 0.13966806829625197
Method: sc, Dims: 1500, Iters: 10000, Average EL rank score over test set: 0.25352799197463954
Method: sc, Dims: 2000, Iters: 10000, Average EL rank score over test set: 0.03470407653110756
Method: sc, Dims: 2500, Iters: 10000, Average EL rank score over test set: 0.02899044470029293
Method: counting, Dims: 1000, Iters: 10000, Average EL rank score over test set: 0.007356847657206943
Method: counting, Dims: 1500, Iters: 10000, Average EL rank score over test set: 0.0045372992891176776
Method: counting, Dims: 2000, Iters: 10000, Average EL rank score over test set: 0.001872459454534864
Method: counting, Dims: 2500, Iters: 10000, Average EL rank score over test set: 0.00374249454226948


In [51]:
from tucker_tensor import _voc_index
total_score = 0
for tup in test_set:
    print(f"Evaluating tuple: {tup}")
    v, s, o = tensor.fetch_latents(tup)
    score = 0
    for i, element in enumerate(tup):
        role = ["verb", "subject", "object"][i]
        r2i = _voc_index(role)
        G_excluded = tensor.excluded_role_vector(tup, role=role)
        F = tensor.factors[i].cpu().numpy()    # (N,R)
        similarities = F @ G_excluded / (np.linalg.norm(F, axis=1) * np.linalg.norm(G_excluded))
        # we print the rank of the actual element
        idx = tensor.vocab[r2i][element]
        rank = np.sum(similarities >= similarities[idx])
        print(f"Rank of the correct {role} '{element}' for excluded role vector: {rank} out of {len(similarities)}")
        score += 1/rank
        print(score)
    print(f"Score: {score/len(tup)}")
    total_score += score/len(tup)
    print()
print(f"Total score over test set: {total_score/len(test_set)}")

Evaluating tuple: ('voeren', 'president', 'oorlog')
Rank of the correct verb 'voeren' for excluded role vector: 1 out of 1000
1.0
Rank of the correct subject 'president' for excluded role vector: 23 out of 1000
1.0434782608695652
Rank of the correct object 'oorlog' for excluded role vector: 2 out of 1000
1.5434782608695652
Score: 0.5144927536231884

Evaluating tuple: ('spelen', 'kind', 'bal')
Rank of the correct verb 'spelen' for excluded role vector: 36 out of 1000
0.027777777777777776
Rank of the correct subject 'kind' for excluded role vector: 146 out of 1000
0.03462709284627093
Rank of the correct object 'bal' for excluded role vector: 70 out of 1000
0.04891280713198521
Score: 0.01630426904399507

Evaluating tuple: ('lezen', 'student', 'boek')
Rank of the correct verb 'lezen' for excluded role vector: 48 out of 1000
0.020833333333333332
Rank of the correct subject 'student' for excluded role vector: 424 out of 1000
0.023191823899371068
Rank of the correct object 'boek' for excluded

Evaluating tuple: ('voeren', 'president', 'oorlog')
Rank of the correct verb 'voeren' for excluded role vector: 1 out of 1000
Rank of the correct subject 'president' for excluded role vector: 23 out of 1000
Rank of the correct object 'oorlog' for excluded role vector: 2 out of 1000

Evaluating tuple: ('spelen', 'kind', 'bal')
Rank of the correct verb 'spelen' for excluded role vector: 36 out of 1000
Rank of the correct subject 'kind' for excluded role vector: 146 out of 1000
Rank of the correct object 'bal' for excluded role vector: 70 out of 1000

Evaluating tuple: ('lezen', 'student', 'boek')
Rank of the correct verb 'lezen' for excluded role vector: 48 out of 1000
Rank of the correct subject 'student' for excluded role vector: 424 out of 1000
Rank of the correct object 'boek' for excluded role vector: 41 out of 1000

Evaluating tuple: ('rijden', 'chauffeur', 'auto')
Rank of the correct verb 'rijden' for excluded role vector: 62 out of 1000
Rank of the correct subject 'chauffeur' for excluded role vector: 76 out of 1000
Rank of the correct object 'auto' for excluded role vector: 148 out of 1000
